# evaglass spot · Blender render (Kaggle GPU)

Kaggle'da **Settings → Accelerator: GPU T4 x2** (veya P100) ve **Internet: On** seçin. Haftalık ücretsiz GPU kotası ~30 saat.

Bu defter: Blender 4.2 LTS'i indirir, repodaki `spot/blender/evaglass_spot.py` sahnesini OPTIX ile render eder, PNG dizilerini ffmpeg ile mp4 yapar ve `/kaggle/working/` altına koyar.

Gerçek gözlük modeli ve ekran kayıtlarınızı Kaggle Dataset olarak ekleyip aşağıdaki `GLASSES`, `SCREEN_PHONE`, `SCREEN_WATCH` yollarını doldurun. Ekran kayıtları doğrudan mp4 olabilir. Dataset adı önerisi: `evaglass-assets`, içinde `phone.mp4`, `watch.mp4`. Boş kalırsa yer tutucu geometri ve düz renkli ekranlar kullanılır.

In [ ]:
!nvidia-smi
import os, subprocess
BRANCH = "claude/highfield-senior-animator-k39icv"
REPO   = "https://github.com/evatechnosoft/evaglass-releases"
SHOTS  = ["SH010", "SH050", "SH060"]
SAMPLES = 64          # 64 temiz; hizli deneme icin 16
RES     = 100         # yuzde
GLASSES      = ""     # orn: /kaggle/input/evaglass-assets/glasses.glb
SCREEN_PHONE = ""     # orn: /kaggle/input/evaglass-assets/phone.mp4  (mp4 dogrudan okunur)
SCREEN_WATCH = ""     # orn: /kaggle/input/evaglass-assets/watch.mp4

In [ ]:
%cd /kaggle/working
!rm -rf evaglass-releases && git clone -q --depth 1 -b $BRANCH $REPO
BL = "blender-4.2.3-linux-x64"
if not os.path.isdir(BL):
    !wget -q https://download.blender.org/release/Blender4.2/{BL}.tar.xz && tar xf {BL}.tar.xz && rm {BL}.tar.xz
!./{BL}/blender --version | head -1

In [ ]:
extra = []
if GLASSES: extra += ["--glasses", GLASSES]
if SCREEN_PHONE: extra += ["--screen-phone", SCREEN_PHONE]
if SCREEN_WATCH: extra += ["--screen-watch", SCREEN_WATCH]
# once her plandan tek kare test (hizli kontrol)
for sh in SHOTS:
    subprocess.run([f"./{BL}/blender","-b","-P","evaglass-releases/spot/blender/evaglass_spot.py","--",
                    "--shot",sh,"--frame","40","--res","50","--samples","16","--device","OPTIX","--out","/kaggle/working/test"]+extra, check=True)
from IPython.display import Image, display
import glob
for f in sorted(glob.glob("/kaggle/working/test/*/test_*.png")): display(Image(f, width=640))

In [ ]:
# tam render: T4 uzerinde plan basina yaklasik 10-25 dk (64 sample, 1080p)
for sh in SHOTS:
    subprocess.run([f"./{BL}/blender","-b","-P","evaglass-releases/spot/blender/evaglass_spot.py","--",
                    "--shot",sh,"--res",str(RES),"--samples",str(SAMPLES),"--device","OPTIX","--out","/kaggle/working/out"]+extra, check=True)
    !ffmpeg -y -loglevel error -framerate 24 -i /kaggle/working/out/{sh}/frame_%04d.png -c:v libx264 -pix_fmt yuv420p -crf 18 /kaggle/working/{sh}.mp4
!ls -la /kaggle/working/*.mp4

Çıktılar: `/kaggle/working/SH010.mp4`, `SH050.mp4`, `SH060.mp4`. Sağdaki **Output** panelinden indirin. Kurgu için DaVinci Resolve veya Kdenlive'a alın; SH020 ve SH030 telefon/saat ekran kaydı, SH040 ise `02_wan22_broll.ipynb` çıktısı veya telefonla çekilmiş gerçek plan.